In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

In [3]:
global_retail_df = pd.read_csv('/content/drive/MyDrive/ML_Project_2.0/transactions.csv')
print(global_retail_df.head())

            Invoice ID  Line  Customer ID  Product ID Size    Color  \
0  INV-US-001-03558761     1        47162         485    M      NaN   
1  INV-US-001-03558761     2        47162        2779    G      NaN   
2  INV-US-001-03558761     3        47162          64    M  NEUTRAL   
3  INV-US-001-03558762     1        10142         131    M     BLUE   
4  INV-US-001-03558762     2        10142         716    L    WHITE   

   Unit Price  Quantity                 Date  Discount  Line Total  Store ID  \
0        80.5         1  2023-01-01 15:42:00       0.0        80.5         1   
1        31.5         1  2023-01-01 15:42:00       0.4        18.9         1   
2        45.5         1  2023-01-01 15:42:00       0.4        27.3         1   
3        70.0         1  2023-01-01 20:04:00       0.4        42.0         1   
4        26.0         1  2023-01-01 20:04:00       0.0        26.0         1   

   Employee ID Currency Currency Symbol               SKU Transaction Type  \
0            7

In [4]:
true_dupe_mask = global_retail_df.duplicated(subset=['Invoice ID', 'Line'], keep=False)
true_dupes = global_retail_df[true_dupe_mask]
print("Rows involved in true invoice+line duplication:", true_dupes.shape[0])
true_dupes.sort_values(['Invoice ID','Line']).head(10)

Rows involved in true invoice+line duplication: 42682


,Invoice ID,Line,Customer ID,Product ID,Size,Color,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,SKU,Transaction Type,Payment Method,Invoice Total
1687960,RET-CN-006-02865002,1,361068,1795,XL,BLACK,367.0,1,2023-01-01 00:00:00,0.0,-220.2,6,67,CNY,¥,FECO1795-XL-BLACK,Return,Credit Card,-220.2
1692438,RET-CN-006-02865002,1,361068,1795,XL,BLACK,367.0,1,2023-01-16 00:00:00,0.0,-220.2,6,67,CNY,¥,FECO1795-XL-BLACK,Return,Credit Card,-220.2
1688063,RET-CN-006-02865012,1,402687,105,S,YELLOW,346.5,1,2023-01-01 00:00:00,0.0,-207.9,6,67,CNY,¥,FECO105-S-YELLOW,Return,Cash,-207.9
1691200,RET-CN-006-02865012,1,402687,105,S,YELLOW,346.5,1,2023-01-10 00:00:00,0.0,-207.9,6,67,CNY,¥,FECO105-S-YELLOW,Return,Cash,-207.9
1688651,RET-CN-006-02865128,1,388929,2177,M,NaN,122.5,1,2023-01-01 00:00:00,0.0,-122.5,6,72,CNY,¥,MAUN2177-M-,Return,Credit Card,-253.6
1691906,RET-CN-006-02865128,1,388929,2177,M,NaN,122.5,1,2023-01-14 00:00:00,0.0,-122.5,6,72,CNY,¥,MAUN2177-M-,Return,Credit Card,-122.5
1689678,RET-CN-006-02865290,1,387738,949,M,PINK,346.5,1,2023-01-05 00:00:00,0.0,-207.9,6,72,CNY,¥,MASW949-M-PINK,Return,Credit Card,-883.4
1692607,RET-CN-006-02865290,1,387738,949,M,PINK,346.5,1,2023-01-17 00:00:00,0.0,-207.9,6,72,CNY,¥,MASW949-M-PINK,Return,Credit Card,-883.4
1689679,RET-CN-006-02865290,2,387738,173,M,NaN,483.5,1,2023-01-05 00:00:00,0.0,-483.5,6,72,CNY,¥,MASU173-M-,Return,Credit Card,-883.4
1692608,RET-CN-006-02865290,2,387738,173,M,NaN,483.5,1,2023-01-17 00:00:00,0.0,-483.5,6,72,CNY,¥,MASU173-M-,Return,Credit Card,-883.4


In [7]:
before = global_retail_df.shape[0]
global_retail_df= global_retail_df.drop_duplicates(subset=['Invoice ID', 'Line'], keep='first')
after = global_retail_df.shape[0]
print(f"Dropped {before - after} duplicate rows. New shape: {global_retail_df.shape}")

Dropped 22312 duplicate rows. New shape: (6394515, 19)


In [8]:
dups_all = global_retail_df[global_retail_df.duplicated(keep=False)]
dups_all.sort_values(by=list(global_retail_df.columns)).head(20)

,Invoice ID,Line,Customer ID,Product ID,Size,Color,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,SKU,Transaction Type,Payment Method,Invoice Total


In [9]:

dedup_cols = [c for c in global_retail_df.columns if c != 'Line']

all_dupes = global_retail_df[global_retail_df.duplicated(subset=dedup_cols, keep=False)].copy()


all_dupes = all_dupes.sort_values(['Invoice ID'] + dedup_cols[1:] + ['Line'])


all_dupes['Record_Type'] = np.where(
    all_dupes.duplicated(subset=dedup_cols, keep='first'), 'Duplicate', 'Original'
)

print("Total rows involved:", all_dupes.shape[0])
print(all_dupes['Record_Type'].value_counts())
print(all_dupes.groupby(['Transaction Type', 'Record_Type']).size())

all_dupes.head(20)

Total rows involved: 1106
Record_Type
Original     553
Duplicate    553
Name: count, dtype: int64
Transaction Type  Record_Type
Return            Duplicate       24
                  Original        24
Sale              Duplicate      529
                  Original       529
dtype: int64


,Invoice ID,Line,Customer ID,Product ID,Size,Color,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,SKU,Transaction Type,Payment Method,Invoice Total,Record_Type
1696850,INV-CN-006-02871204,3,357683,2771,M,NaN,157.5,1,2023-02-05 17:26:00,0.00,157.50,6,68,CNY,¥,MASH2771-M-,Sale,Credit Card,1561.50,Original
1696851,INV-CN-006-02871204,4,357683,2771,M,NaN,157.5,1,2023-02-05 17:26:00,0.00,157.50,6,68,CNY,¥,MASH2771-M-,Sale,Credit Card,1561.50,Duplicate
1710019,INV-CN-006-02880050,1,388960,4137,XL,NaN,535.0,1,2023-03-18 14:06:00,0.35,347.75,6,72,CNY,¥,FEDR4137-XL-,Sale,Credit Card,2376.35,Original
1710022,INV-CN-006-02880050,4,388960,4137,XL,NaN,535.0,1,2023-03-18 14:06:00,0.35,347.75,6,72,CNY,¥,FEDR4137-XL-,Sale,Credit Card,2376.35,Duplicate
1721008,INV-CN-006-02887465,1,364239,5685,M,NaN,796.0,1,2023-04-07 16:41:00,0.00,796.00,6,68,CNY,¥,MASU5685-M-,Sale,Credit Card,2027.50,Original
1721011,INV-CN-006-02887465,4,364239,5685,M,NaN,796.0,1,2023-04-07 16:41:00,0.00,796.00,6,68,CNY,¥,MASU5685-M-,Sale,Credit Card,2027.50,Duplicate
1725589,INV-CN-006-02890520,2,373964,5519,M,NaN,276.5,1,2023-04-23 15:57:00,0.00,276.50,6,72,CNY,¥,FESK5519-M-,Sale,Credit Card,3201.00,Original
1725592,INV-CN-006-02890520,5,373964,5519,M,NaN,276.5,1,2023-04-23 15:57:00,0.00,276.50,6,72,CNY,¥,FESK5519-M-,Sale,Credit Card,3201.00,Duplicate
1744823,INV-CN-006-02903457,1,391986,3526,38,NaN,846.0,1,2023-06-20 17:48:00,0.00,846.00,6,71,CNY,¥,MAPA3526-38-,Sale,Credit Card,4251.50,Original
1744827,INV-CN-006-02903457,5,391986,3526,38,NaN,846.0,1,2023-06-20 17:48:00,0.00,846.00,6,71,CNY,¥,MAPA3526-38-,Sale,Credit Card,4251.50,Duplicate


In [ ]:
global_retail_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6416827 entries, 0 to 6416826
Data columns (total 19 columns):
 #   Column            Dtype  
---  ------            -----  
 0   Invoice ID        object 
 1   Line              int64  
 2   Customer ID       int64  
 3   Product ID        int64  
 4   Size              object 
 5   Color             object 
 6   Unit Price        float64
 7   Quantity          int64  
 8   Date              object 
 9   Discount          float64
 10  Line Total        float64
 11  Store ID          int64  
 12  Employee ID       int64  
 13  Currency          object 
 14  Currency Symbol   object 
 15  SKU               object 
 16  Transaction Type  object 
 17  Payment Method    object 
 18  Invoice Total     float64
dtypes: float64(4), int64(6), object(9)
memory usage: 930.2+ MB


In [ ]:
global_retail_df.isnull().sum()

,0
Invoice ID,0
Line,0
Customer ID,0
Product ID,0
Size,413102
Color,4350783
Unit Price,0
Quantity,0
Date,0
Discount,0


In [ ]:
global_retail_df = global_retail_df.drop(columns=['Size', 'Color', 'Payment Method', 'Invoice Total', 'SKU'])
display(global_retail_df.head())

,Invoice ID,Line,Customer ID,Product ID,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,Transaction Type
0,INV-US-001-03558761,1,47162,485,80.5,1,2023-01-01 15:42:00,0.0,80.5,1,7,USD,$,Sale
1,INV-US-001-03558761,2,47162,2779,31.5,1,2023-01-01 15:42:00,0.4,18.9,1,7,USD,$,Sale
2,INV-US-001-03558761,3,47162,64,45.5,1,2023-01-01 15:42:00,0.4,27.3,1,7,USD,$,Sale
3,INV-US-001-03558762,1,10142,131,70.0,1,2023-01-01 20:04:00,0.4,42.0,1,6,USD,$,Sale
4,INV-US-001-03558762,2,10142,716,26.0,1,2023-01-01 20:04:00,0.0,26.0,1,6,USD,$,Sale


In [ ]:
return_transactions_df = global_retail_df[global_retail_df['Transaction Type'] == 'Return']
display(return_transactions_df.head())

,Invoice ID,Line,Customer ID,Product ID,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,Transaction Type
15,RET-US-001-03558764,1,1968,1208,75.0,1,2023-01-01 00:00:00,0.0,-45.0,1,9,USD,$,Return
16,RET-US-001-03558764,2,1968,2639,45.5,1,2023-01-01 00:00:00,0.0,-27.3,1,9,USD,$,Return
31,RET-US-001-03558763,1,8286,1395,95.5,1,2023-01-01 00:00:00,0.0,-95.5,1,6,USD,$,Return
40,RET-US-001-03558767,1,14906,2924,54.0,1,2023-01-01 00:00:00,0.0,-32.4,1,13,USD,$,Return
103,RET-US-001-03558799,1,21979,377,42.0,1,2023-01-01 00:00:00,0.0,-25.2,1,7,USD,$,Return


In [ ]:
print(len(return_transactions_df))

339627


In [ ]:
print(return_transactions_df['Discount'].describe())
print(return_transactions_df['Quantity'].describe())

count    339627.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: Discount, dtype: float64
count    339627.000000
mean          1.101052
std           0.398044
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           3.000000
Name: Quantity, dtype: float64


In [ ]:
sale_transactions_df = global_retail_df[global_retail_df['Transaction Type'] == 'Sale']
display(sale_transactions_df.head())

,Invoice ID,Line,Customer ID,Product ID,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,Transaction Type
0,INV-US-001-03558761,1,47162,485,80.5,1,2023-01-01 15:42:00,0.0,80.5,1,7,USD,$,Sale
1,INV-US-001-03558761,2,47162,2779,31.5,1,2023-01-01 15:42:00,0.4,18.9,1,7,USD,$,Sale
2,INV-US-001-03558761,3,47162,64,45.5,1,2023-01-01 15:42:00,0.4,27.3,1,7,USD,$,Sale
3,INV-US-001-03558762,1,10142,131,70.0,1,2023-01-01 20:04:00,0.4,42.0,1,6,USD,$,Sale
4,INV-US-001-03558762,2,10142,716,26.0,1,2023-01-01 20:04:00,0.0,26.0,1,6,USD,$,Sale


In [ ]:
print(sale_transactions_df['Quantity'].describe())

count    6.077200e+06
mean     1.100198e+00
std      3.962860e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      3.000000e+00
Name: Quantity, dtype: float64


In [ ]:
print(sale_transactions_df['Discount'].describe())

count    6.077200e+06
mean     1.257114e-01
std      2.024585e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      2.500000e-01
max      6.000000e-01
Name: Discount, dtype: float64


In [ ]:
duplicate_count = global_retail_df.duplicated().sum()
print(duplicate_count)

1371


In [ ]:
global_retail_df[global_retail_df.duplicated()]

,Invoice ID,Line,Customer ID,Product ID,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,Transaction Type
1306,RET-US-001-03558990,1,45178,2602,25.5,1,2023-01-01 00:00:00,0.0,-15.30,1,13,USD,$,Return
1415,RET-US-001-03558814,1,1350,1873,46.0,1,2023-01-01 00:00:00,0.0,-27.60,1,5,USD,$,Return
3134,RET-US-001-03559999,3,210055,1297,33.0,1,2023-01-01 00:00:00,0.0,-19.80,1,12,USD,$,Return
5967,RET-US-001-03562782,1,12413,1561,35.0,1,2023-01-07 00:00:00,0.0,-21.00,1,7,USD,$,Return
7471,RET-US-001-03562412,1,26019,2104,34.0,1,2023-01-08 00:00:00,0.0,-20.40,1,13,USD,$,Return
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6402429,RET-PT-035-01486275,1,1638732,13579,36.5,1,2024-12-22 00:00:00,0.0,-36.50,35,400,EUR,€,Return
6402430,RET-PT-035-01486275,2,1638732,12641,37.0,1,2024-12-22 00:00:00,0.0,-37.00,35,400,EUR,€,Return
6404980,RET-PT-035-01488077,1,1643125,12421,30.5,1,2024-12-25 00:00:00,0.0,-15.25,35,401,EUR,€,Return
6405025,RET-PT-035-01488447,1,1639996,14003,36.0,1,2024-12-25 00:00:00,0.0,-18.00,35,399,EUR,€,Return


In [ ]:
global_retail_df['Date'] = pd.to_datetime(global_retail_df['Date'])
print(global_retail_df['Date'].dtype)

datetime64[ns]


In [ ]:
global_retail_df.loc[[global_retail_df["Date"].idxmax(), global_retail_df["Date"].idxmin()]]

,Invoice ID,Line,Customer ID,Product ID,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,Transaction Type
476841,INV-US-001-03881129,1,10315,17148,66.0,1,2025-03-18 20:59:00,0.35,42.9,1,11,USD,$,Sale
15,RET-US-001-03558764,1,1968,1208,75.0,1,2023-01-01 00:00:00,0.00,-45.0,1,9,USD,$,Return


In [ ]:
global_retail_df.head(30)

,Invoice ID,Line,Customer ID,Product ID,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,Transaction Type
0,INV-US-001-03558761,1,47162,485,80.5,1,2023-01-01 15:42:00,0.0,80.5,1,7,USD,$,Sale
1,INV-US-001-03558761,2,47162,2779,31.5,1,2023-01-01 15:42:00,0.4,18.9,1,7,USD,$,Sale
2,INV-US-001-03558761,3,47162,64,45.5,1,2023-01-01 15:42:00,0.4,27.3,1,7,USD,$,Sale
3,INV-US-001-03558762,1,10142,131,70.0,1,2023-01-01 20:04:00,0.4,42.0,1,6,USD,$,Sale
4,INV-US-001-03558762,2,10142,716,26.0,1,2023-01-01 20:04:00,0.0,26.0,1,6,USD,$,Sale
5,INV-US-001-03558762,3,10142,644,9.0,1,2023-01-01 20:04:00,0.0,9.0,1,6,USD,$,Sale
6,INV-US-001-03558763,1,8286,1395,95.5,1,2023-01-01 20:47:00,0.0,95.5,1,6,USD,$,Sale
7,INV-US-001-03558764,1,1968,1208,75.0,1,2023-01-01 08:06:00,0.4,45.0,1,9,USD,$,Sale
8,INV-US-001-03558764,2,1968,2639,45.5,1,2023-01-01 08:06:00,0.4,27.3,1,9,USD,$,Sale
9,INV-US-001-03558765,1,265336,1822,32.0,1,2023-01-01 17:25:00,0.4,19.2,1,12,USD,$,Sale


In [ ]:
true_dupe_mask = global_retail_df.duplicated(subset=['Invoice ID', 'Line'], keep=False)
true_dupes = global_retail_df[true_dupe_mask]
print("Rows involved in true invoice+line duplication:", true_dupes.shape[0])
true_dupes.sort_values(['Invoice ID','Line']).head(10)

Rows involved in true invoice+line duplication: 42682


,Invoice ID,Line,Customer ID,Product ID,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,Transaction Type
1687960,RET-CN-006-02865002,1,361068,1795,367.0,1,2023-01-01,0.0,-220.2,6,67,CNY,¥,Return
1692438,RET-CN-006-02865002,1,361068,1795,367.0,1,2023-01-16,0.0,-220.2,6,67,CNY,¥,Return
1688063,RET-CN-006-02865012,1,402687,105,346.5,1,2023-01-01,0.0,-207.9,6,67,CNY,¥,Return
1691200,RET-CN-006-02865012,1,402687,105,346.5,1,2023-01-10,0.0,-207.9,6,67,CNY,¥,Return
1688651,RET-CN-006-02865128,1,388929,2177,122.5,1,2023-01-01,0.0,-122.5,6,72,CNY,¥,Return
1691906,RET-CN-006-02865128,1,388929,2177,122.5,1,2023-01-14,0.0,-122.5,6,72,CNY,¥,Return
1689678,RET-CN-006-02865290,1,387738,949,346.5,1,2023-01-05,0.0,-207.9,6,72,CNY,¥,Return
1692607,RET-CN-006-02865290,1,387738,949,346.5,1,2023-01-17,0.0,-207.9,6,72,CNY,¥,Return
1689679,RET-CN-006-02865290,2,387738,173,483.5,1,2023-01-05,0.0,-483.5,6,72,CNY,¥,Return
1692608,RET-CN-006-02865290,2,387738,173,483.5,1,2023-01-17,0.0,-483.5,6,72,CNY,¥,Return


In [ ]:
print(true_dupes['Transaction Type'].value_counts())

Transaction Type
Return    42682
Name: count, dtype: int64


In [ ]:
print(global_retail_df['Currency'].value_counts())

Currency
EUR    2549397
USD    1687764
CNY    1553438
GBP     626228
Name: count, dtype: int64


In [ ]:
print(global_retail_df['Unit Price'].describe())

count    6.416827e+06
mean     1.324640e+02
std      1.850971e+02
min      2.000000e+00
25%      3.250000e+01
50%      5.100000e+01
75%      1.165000e+02
max      1.153500e+03
Name: Unit Price, dtype: float64


In [ ]:
print("Unique customers:", global_retail_df['Customer ID'].nunique())
print("Unique invoices:", global_retail_df['Invoice ID'].nunique())

Unique customers: 1283707
Unique invoices: 4540404


In [ ]:
fx_to_usd = {
    'USD': 1.0,
    'EUR': 1.08,
    'GBP': 1.27,
    'CNY': 0.14,
}

global_retail_df['FX_Rate'] = global_retail_df['Currency'].map(fx_to_usd)

global_retail_df['Unit Price'] = global_retail_df['Unit Price'] * global_retail_df['FX_Rate']
global_retail_df['Line Total'] = global_retail_df['Line Total'] * global_retail_df['FX_Rate']

global_retail_df['Currency'] = 'USD'
global_retail_df['Currency Symbol'] = '$'

global_retail_df = global_retail_df.drop(columns=['FX_Rate'])

In [ ]:
global_retail_df.head(10)

,Invoice ID,Line,Customer ID,Product ID,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,Transaction Type
0,INV-US-001-03558761,1,47162,485,80.5,1,2023-01-01 15:42:00,0.0,80.5,1,7,USD,$,Sale
1,INV-US-001-03558761,2,47162,2779,31.5,1,2023-01-01 15:42:00,0.4,18.9,1,7,USD,$,Sale
2,INV-US-001-03558761,3,47162,64,45.5,1,2023-01-01 15:42:00,0.4,27.3,1,7,USD,$,Sale
3,INV-US-001-03558762,1,10142,131,70.0,1,2023-01-01 20:04:00,0.4,42.0,1,6,USD,$,Sale
4,INV-US-001-03558762,2,10142,716,26.0,1,2023-01-01 20:04:00,0.0,26.0,1,6,USD,$,Sale
5,INV-US-001-03558762,3,10142,644,9.0,1,2023-01-01 20:04:00,0.0,9.0,1,6,USD,$,Sale
6,INV-US-001-03558763,1,8286,1395,95.5,1,2023-01-01 20:47:00,0.0,95.5,1,6,USD,$,Sale
7,INV-US-001-03558764,1,1968,1208,75.0,1,2023-01-01 08:06:00,0.4,45.0,1,9,USD,$,Sale
8,INV-US-001-03558764,2,1968,2639,45.5,1,2023-01-01 08:06:00,0.4,27.3,1,9,USD,$,Sale
9,INV-US-001-03558765,1,265336,1822,32.0,1,2023-01-01 17:25:00,0.4,19.2,1,12,USD,$,Sale


In [ ]:
print(global_retail_df['Currency'].value_counts())

Currency
USD    6416827
Name: count, dtype: int64


In [ ]:
global_retail_df.to_csv("/content/drive/MyDrive/ML_Project_2.0/new_trasaction.csv", index=False)
print(f"Saved to customer_rfm_features.csv with shape {global_retail_df.shape}")

Saved to customer_rfm_features.csv with shape (6416827, 14)
